# GPT 图像生成模型提示指南

## 1. 简介

OpenAI 的 gpt-image 生成模型专为生产级视觉效果和高度可控的创意工作流而设计。它们既适合专业设计任务，也适合迭代式内容创作；根据工作流的不同，既支持高质量渲染，也支持低延迟用例。

关键能力包括： 

- **高保真照片级写实**，具备自然光照、准确的材质和丰富的色彩呈现
- **灵活的质量–延迟权衡**，在较低设置下生成更快，同时视觉质量仍超越上一代图像模型
- **稳健的面部与身份保持**，适用于编辑、角色一致性和多步骤工作流
- **透明背景的 PNG 和 WebP 资产（预览版）**，借助 `gpt-image-2` 可用于可复用的 logo、产品抠图、贴纸和演示图形
- **可靠的文字渲染**，图像中的文字清晰锐利、版式一致、对比度强
- **复杂的结构化视觉内容**，包括信息图、示意图和多格构图
- **精确的风格控制和风格迁移**，只需极少提示，支持从品牌设计系统到美术风格的各种需求
- **强大的现实世界知识和推理能力**，能够准确描绘物体、环境和场景

本指南重点介绍针对 `gpt-image-2` 的提示模式、最佳实践以及取自真实生产用例的示例提示。它是我们能力最强的图像模型，图像质量更高、编辑性能更好，并更广泛地支持生产工作流。`low` 质量设置在对延迟敏感的用例中表现尤为出色，而当最高保真度至关重要时，`medium` 和 `high` 仍然是很好的选择。 


## 1.1 OpenAI 图像模型参数 

本节是本指南所涵盖图像模型的参考资料，重点包括：

- 模型名称
- 支持的 `outputQuality` 值
- 支持的 `input_fidelity` 值
- 支持的 `size` / 分辨率行为
- 透明资产对 `background` 和 `output_format` 的要求
- 按工作流推荐的用例

## 模型概览
截至 2026 年 4 月 21 日，OpenAI 提供以下图像模型。

| 模型 | `outputQuality` | `input_fidelity` | 分辨率 | 推荐用途 |
| --- | --- | --- | --- | --- |
| `gpt-image-2` | `low`, `medium`, `high` | 已禁用。`input_fidelity` 对该模型不起作用，因为其输出默认就是高保真的 | 满足下述约束条件的任意分辨率 | 新项目的推荐默认选择。适用于最高质量的生成和编辑、文字密集的图像、照片级写实、合成、对身份敏感的编辑，以及减少重试比追求最低成本更重要的工作流。 |
| `gpt-image-1.5` | `low`, `medium`, `high` | `low`, `high` | `1024x1024`, `1024x1536`, `1536x1024`, `auto` | 在迁移期间保留给已验证的现有工作流。对于新工作，优先使用 `gpt-image-2`，尤其是在质量、编辑可靠性或灵活尺寸很重要时。 |
| `gpt-image-1` | `low`, `medium`, `high` | `low`, `high` | `1024x1024`, `1024x1536`, `1536x1024`, `auto` | 仅用于旧版兼容。如果你正在启动新工作流或更新提示，请迁移到 `gpt-image-2`；仅在验证升级期间需要短期稳定性时才保留 `gpt-image-1`。 |
| `gpt-image-1-mini` | `low`, `medium`, `high` | `low`, `high` | `1024x1024`, `1024x1536`, `1536x1024`, `auto` | 当成本和吞吐量是主要约束时使用：大批量变体生成、快速构思、预览、轻量级个性化，以及不需要最强生成或编辑性能的草稿资产。 |

### 使用 `gpt-image-2` 生成透明背景（预览版）

`gpt-image-2` 的透明背景功能目前处于预览阶段。要生成或编辑带透明背景的图像：

- 设置 `background="transparent"`。
- 设置 `output_format="png"`（默认值）或 `output_format="webp"`。两者都支持透明；`jpeg` 不支持。
- PNG 输出时省略 `output_compression`。WebP 支持可选压缩。
- 明确要求在完全透明的背景上呈现一个独立主体，不要有场景、纯色背景、棋盘格或多余的阴影。
- 进行编辑时，在每个提示中都明确要求保留透明背景，以免后续步骤引入新的背景。
- 将返回的图像保存为 PNG 或 WebP 文件，并在整个下游处理过程中保留其 alpha 通道。

### `gpt-image-2` 尺寸选项

`gpt-image-2` 支持通过 `size` 参数传入的任意分辨率，只要同时满足以下所有约束：

- 最长边必须小于 `3840px`
- 两条边都必须是 `16` 的倍数
- 长边与短边之比不得大于 `3:1`
- 总像素数不得超过 `8,294,400`
- 总像素数不得少于 `655,360`

如果输出图像超过 `2560x1440` 像素（总计 `3,686,400` 像素，通常称为 2K），请将其视为实验性功能，因为超过该尺寸后结果可能更不稳定。

### 常用的 `gpt-image-2` 尺寸

以下是一些符合上述约束的实用参考尺寸：

| 标签 | 分辨率 | 说明 |
| --- | --- | --- |
| 高清竖版 | `1024x1536` | 标准竖版选项 |
| 高清横版 | `1536x1024` | 标准横版选项 |
| 正方形 | `1024x1024` | 良好的通用默认值 |
| 2K / QHD | `2560x1440` | 流行的宽屏格式，也是 `gpt-image-2` 推荐的可靠性上限 |
| 4K / UHD | `3840x2160` | 实验性的高端目标。如果最长边规则被严格执行为 `< 3840`，请向下取整到最接近的有效尺寸，例如 `3824x2144` |

### 何时使用哪个模型

- 对于大多数生产工作流，默认选择 `gpt-image-2`。它是整体最强的模型，也是目前使用 `gpt-image-1.5` 或 `gpt-image-1` 生成高质量输出的团队的理想升级目标。
- 当速度和单位成本是决策的主导因素时，选择 quality 为 low 的 `gpt-image-2`。该设置在许多用例中都有不错的质量，非常适合大批量生成和实验。对于这些用例，你也可以尝试 `gpt-image-1-mini`，但我们发现 quality: low 的效果同样好。
- 仅在验证提示迁移、对输出进行回归测试，或维护尚未准备好迁移的旧工作流时，为了向后兼容而保留 `gpt-image-1.5` 或 `gpt-image-1`。

### 从 `gpt-image-1.5` 和 `gpt-image-1` 升级的推荐路径

对于目前使用 `gpt-image-1.5` 或 `gpt-image-1` 的工作流，建议如下：

- 对于面向客户的资产、照片级写实生成、编辑密集型流程、对品牌敏感的创意、图中文字工作，以及任何首次生成质量更好就能减少人工审核或重跑的工作流，升级到 `gpt-image-2`。
- 仅当主要目标是降低大批量探索性或低风险图像的成本时，才考虑使用 `gpt-image-1-mini` 代替旧版模型。
- 在迁移期间，起初尽量保持提示基本不变，在你针对真实工作负载比较过输出质量、延迟和重试率之后，再进行重新调优。

## 2. 提示基础

以下提示基础适用于 GPT 图像生成模型。它们基于 alpha 测试中在生成、编辑、信息图、广告、人物图像、UI 原型和合成等工作流中反复出现的模式。

* **结构 + 目标：** 按一致的顺序编写提示（背景/场景 → 主体 → 关键细节 → 约束），并注明预期用途（广告、UI 原型、信息图），以设定“模式”和精修程度。对于复杂请求，使用带标签的简短段落或换行，而不是一个长段落。

* **提示格式：** 使用最易于维护的格式。只要意图和约束清晰，极简提示、描述性段落、类 JSON 结构、指令式提示和基于标签的提示都能很好地工作。对于生产系统，优先选择便于快速浏览的模板，而不是花哨的提示语法。

* **具体性 + 质量提示词：** 具体说明材质、形状、纹理和视觉媒介（照片、水彩、3D 渲染），仅在需要时添加有针对性的“质量杠杆”（例如 *film grain*、*textured brushstrokes*、*macro detail*）。对于照片级写实，直接在提示中加入“photorealistic”一词，以强力启用模型的照片级写实模式。类似的短语，如“real photograph”“taken on a real camera”“professional photography”或“iPhone photo”也有帮助，但详细的相机参数可能会被宽松地解读，因此主要用它们来控制整体观感和构图，而不是精确的物理模拟。

* **延迟与保真度：** 对于延迟敏感或大批量的用例，从 `quality="low"` 开始，评估它是否满足你的视觉要求。在许多情况下，它能提供足够的保真度，同时生成速度显著更快。对于小号或密集的文字、细节丰富的信息图、特写肖像、对身份敏感的编辑以及高分辨率输出，请在上线前比较 `medium` 或 `high` 的效果。

* **构图：** 指定取景和视角（特写、广角、俯视）、透视/角度（平视、仰拍）以及光照/氛围（柔和漫射光、黄金时段、高对比度）来控制画面。如果布局很重要，请说明元素位置（例如“logo 位于右上角”“主体居中，左侧留白”）。对于广角、电影感、弱光、雨景或霓虹场景，请补充关于尺度、氛围和色彩的细节，以免模型为了表面写实而牺牲氛围。

* **人物、姿势和动作：** 对于场景中的人物，请描述比例、身体取景、视线以及与物体的互动。例如：“全身可见，包括双脚”“相对于桌子是儿童体型”“低头看着打开的书，而不是看镜头”或“双手自然地握住车把”。这些细节有助于身体比例、动作几何关系和视线方向的准确呈现。

* **约束（要改变什么与保留什么）：** 明确说明排除项和不变项（例如“no watermark”“no extra text”“no logos/trademarks”“保留身份/几何结构/布局/品牌元素”）。对于编辑，使用“只改变 X”+“其他一切保持不变”，并在每次迭代中重复需要保留的内容列表，以减少漂移。如果编辑需要精确到位，还要说明不要改变饱和度、对比度、布局、箭头、标签、相机角度或周围物体。

* **透明背景：** 对于 `gpt-image-2`，透明功能处于预览阶段。将 `background="transparent"` 与 `output_format="png"`（默认值）或 `output_format="webp"` 搭配使用；`jpeg` 不支持透明。将主体描述为位于完全透明背景上的独立对象，并明确排除场景、纯色背景、棋盘格和多余的阴影。进行编辑时，重复说明“保留透明背景”，以免工作流引入或保留不透明的场景。PNG 输出时省略 `output_compression`；WebP 支持可选压缩。

* **图像中的文字：** 将字面文字放在**引号**中或使用**全大写**，并将字体细节（字体风格、大小、颜色、位置）作为约束加以说明。对于不易处理的词（品牌名称、不常见的拼写），逐个字母拼写出来，以提高字符准确性。对于小号文字、信息密集的面板和多字体版式，使用 `medium` 或 `high` 质量。

* **多图像输入：** 通过**索引和描述**引用每个输入（“Image 1: product photo… Image 2: style reference…”），并描述它们之间如何交互（“apply Image 2’s style to Image 1”）。进行合成时，明确说明哪些元素移动到哪里（“put the bird from Image 1 on the elephant in Image 2”）。

* **迭代而非堆砌：** 长提示也可以效果很好，但从一个干净的基础提示开始，再通过每次只改一处的小幅后续提示进行优化（“make lighting warmer”“remove the extra tree”“restore the original background”），调试起来更容易。可以使用“same style as before”或“the subject”等引用来利用上下文，但如果关键细节开始漂移，请重新说明。

## 3. 设置

运行一次即可。它会：
- 创建 API 客户端
- 在 images 文件夹中创建 `output_images/`。 
- 添加一个用于保存 base64 图像的小辅助函数

将编辑时用到的所有参考图像放入 `input_images/`（或更新示例中的路径）。保存透明输出时，请使用与所选输出格式匹配的 `.png` 或 `.webp` 扩展名，并保留返回的图像字节；将 RGBA 图像转换为 RGB 会丢弃其透明背景。

In [ ]:
import os
import base64
from openai import OpenAI

client = OpenAI()

os.makedirs("../../images/input_images", exist_ok=True)
os.makedirs("../../images/output_images", exist_ok=True)

def save_image(result, filename: str) -> None:
    """
    Saves the first returned image to the given filename inside the output_images folder.
    """
    image_base64 = result.data[0].b64_json
    out_path = os.path.join("../../images/output_images", filename)
    with open(out_path, "wb") as f:
        f.write(base64.b64decode(image_base64))

from IPython.display import HTML, Image, display

def display_image_grid(items, width=240):
    cards = []
    for item in items:
        title = item.get("title", "")
        label = f'<div style="font-weight:600;margin-bottom:8px">{title}</div>' if title else ""
        cards.append(
            '<div style="text-align:center">'
            + label
            + f'<img src="{item["path"]}" width="{width}" style="max-width:100%;height:auto;" />'
            + '</div>'
        )
    display(HTML('<div style="display:flex;flex-wrap:wrap;gap:16px;align-items:flex-start">' + ''.join(cards) + '</div>'))


下面的示例使用我们能力最强的图像模型 `gpt-image-2`

## 4. 用例——生成（文本 → 图像）

## 4.1 信息图
使用信息图向特定受众（学生、高管、客户或普通大众）解释结构化信息。示例包括讲解图、海报、带标注的示意图、时间线和“视觉百科”类资产。对于布局密集或图中文字较多的情况，建议将输出生成质量设为 "high"。

In [ ]:
prompt = """
Create a detailed Infographic of the functioning and flow of an automatic coffee machine like a Jura. 
From bean basket, to grinding, to scale, water tank, boiler, etc. 
I'd like to understand technically and visually the flow.
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "infographic_coffee_machine_gpt-image-2.png")

输出图像：

![](../../images/infographic_coffee_machine_gpt-image-2.png)

## 4.2 图像中的翻译

用于将现有设计（广告、UI 截图、包装、信息图）本地化为另一种语言，而无需从头重建布局。关键是保留除文字之外的一切——保持字体风格、位置、间距和层级一致——同时逐字准确地翻译，不添加多余的词，除非必要不重新排版，也不对 logo、图标或图像做任何意外改动。

In [ ]:
prompt = """
Translate the text in the infographic to Spanish. Do not change any other aspect of the image.
"""

result = client.images.edit(
    model="gpt-image-2",
    image=[
        open("../../images/output_images/infographic_coffee_machine_gpt-image-2.png", "rb"),
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)


save_image(result, "infographic_coffee_machine_sp_gpt-image-2.png")

输出图像：

![](../../images/infographic_coffee_machine_sp_gpt-image-2.png)

## 4.3 感觉“自然”的照片级写实图像

要获得可信的照片级写实效果，提示模型时要仿佛正在抓拍一张真实的照片。使用摄影语言（镜头、光照、取景），并明确要求真实的质感（毛孔、皱纹、织物磨损、瑕疵）。避免使用暗示影棚精修或摆拍的词语。当细节很重要时，设置 quality="high"。

In [ ]:
prompt = """
Create a photorealistic candid photograph of an elderly sailor standing on a small fishing boat. 
He has weathered skin with visible wrinkles, pores, and sun texture, and a few faded traditional sailor tattoos on his arms. 
He is calmly adjusting a net while his dog sits nearby on the deck. Shot like a 35mm film photograph, medium close-up at eye level, using a 50mm lens. 
Soft coastal daylight, shallow depth of field, subtle film grain, natural color balance. 
The image should feel honest and unposed, with real skin texture, worn materials, and everyday detail. No glamorization, no heavy retouching. 
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "photorealism-gpt-image-2.png")

输出图像：

![](../../images/photorealism-gpt-image-2.png)

## 4.4 世界知识

GPT 图像生成模型能够将强大的推理能力与世界知识结合起来。例如，当被要求生成一个设定在 1969 年 8 月纽约州贝瑟尔（Bethel）的场景时，它们能够推断出这是伍德斯托克音乐节，并在没有被明确告知该事件的情况下生成准确且符合背景的图像。 

In [ ]:
prompt = """
Create a realistic outdoor crowd scene in Bethel, New York on August 16, 1969.
Photorealistic, period-accurate clothing, staging, and environment.
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "world_knowledge-gpt-image-2.png")

输出图像：

![](../../images/world_knowledge-gpt-image-2.png)

## 4.5 Logo 生成

出色的 logo 生成来自清晰的品牌约束和简洁性。描述品牌的个性和使用场景，然后要求一个干净、原创的标志，具有鲜明的形状、平衡的负空间，并能在各种尺寸下缩放使用。当 logo 需要在不同的网站、营销活动或演示背景中复用时，请使用透明背景的 PNG 或 WebP 文件。

你可以指定参数 "n" 来表示想要生成的变体数量。 

In [ ]:
prompt = """
Create an original, non-infringing logo for a company called Field & Flour, a local bakery. 
The logo should feel warm, simple, and timeless. Use clean, vector-like shapes, a strong silhouette, and balanced negative space. 
Favor simplicity over detail so it reads clearly at small and large sizes. Flat design, minimal strokes, no gradients unless essential. 
Fully transparent background. Deliver a single centered logo with generous padding, clean alpha edges, and no solid backdrop, scenery, checkerboard, or watermark.
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1536",
    quality="medium",
    background="transparent",
    output_format="png",
    n=4,    # Generate 4 versions of the logo
)

# Save all 4 images to separate files
for i, item in enumerate(result.data, start=1):
    image_base64 = item.b64_json
    image_bytes = base64.b64decode(image_base64)
    with open(f"../../images/output_images/logo_generation_{i}_gpt-image-2.png", "wb") as f:
        f.write(image_bytes)

输出图像： 

| 选项 1 | 选项 2 | 选项 3 | 选项 4 |
|:--------:|:--------:|:--------:|:--------:|
| ![](../../images/logo_generation_1_gpt-image-2.png) | ![](../../images/logo_generation_2_gpt-image-2.png) | ![](../../images/logo_generation_3_gpt-image-2.png) | ![](../../images/logo_generation_4_gpt-image-2.png)|

## 4.6 广告生成

当提示写得像一份创意简报，而不是纯粹的技术性图像规格时，广告生成的效果最好。描述品牌、受众、文化、概念、构图和准确的文案，然后让模型在这些边界内做出体现品味的创意决策。这对于早期的营销活动探索很有用，因为模型可以解读受众线索、推断艺术指导方向，并提出让广告显得用心而不只是被渲染出来的视觉细节。

为了获得更好的结果，请在同一个提示中包含品牌定位、期望的氛围、目标受众、场景和标语。如果文字必须出现在图像中，请准确引用原文，并要求干净、清晰易读的字体排印。

In [ ]:
prompt = """
Give me a cool in culture ad / fashion shot for a brand called Thread. 
It's a hip young street brand. The ad shows a group of friends hanging out together with the tagline "Yours to Create."
Make it feel like a polished campaign image for a youth streetwear audience: stylish, contemporary, energetic, and tasteful.
Use clean composition, strong color direction, natural poses, and premium fashion photography cues.
Render the tagline exactly once, clearly and legibly, integrated into the ad layout.
No extra text, no watermarks, no unrelated logos.
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "thread_ad_gpt-image-2.png")

输出图像：

![](../../images/thread_ad_gpt-image-2.png)


## 4.7 故事转连环漫画

对于故事转漫画的生成，将叙事定义为一系列清晰的视觉节拍，每格一个。保持描述具体且聚焦于动作，以便模型能将故事转化为易读、节奏合理的分格画面。

In [ ]:
prompt = """
Create a short vertical comic-style reel with 4 equal-sized panels.
Panel 1: The owner leaves through the front door. The pet is framed in the window behind them, small against the glass, eyes wide, paws pressed high, the house suddenly quiet.
Panel 2: The door clicks shut. Silence breaks. The pet slowly turns toward the empty house, posture shifting, eyes sharp with possibility.
Panel 3: The house transformed. The pet sprawls across the couch like it owns the place, crumbs nearby, sunlight cutting across the room like a spotlight.
Panel 4: The door opens. The pet is seated perfectly by the entrance, alert and composed, as if nothing happened.
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "comic_reel-gpt-image-2.png")

输出图像：

![](../../images/comic_reel-gpt-image-2.png)

## 4.8 UI 原型

当你把产品描述得仿佛它已经存在时，UI 原型的效果最好。重点描述布局、层级、间距和真实的界面元素，避免使用概念艺术类的措辞，这样结果看起来才像一个可用、已上线的界面，而不是设计草图。

In [ ]:
prompt = """
Create a realistic mobile app UI mockup for a local farmers market. 
Show today’s market with a simple header, a short list of vendors with small photos and categories, a small “Today’s specials” section, and basic information for location and hours. 
Design it to be practical, and easy to use. White background, subtle natural accent colors, clear typography, and minimal decoration. 
It should look like a real, well-designed, beautiful app for a small local market. 
Place the UI mockup in an iPhone frame.
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "ui_farmers_market_gpt-image-2.png")

输出图像：

![](../../images/ui_farmers_market_gpt-image-2.png)

## 4.9 科学/教育类视觉内容

科学和教育类视觉内容非常适合生物、化学、课堂讲解、扁平化科学图标体系、示意图和学习资料。像撰写教学设计简报一样编写提示：定义受众、教学目标、视觉格式、必需的标注和科学约束。为获得最佳效果，要求采用干净、扁平的视觉体系，图标风格一致、箭头清晰、标注易读，并留出足够的空白，让学生能快速浏览理解概念。

当准确性很重要时，明确列出必需的组成部分，并说明哪些内容不应包含。对于标注密集的示意图或将用于幻灯片、课程材料的资产，请使用 `quality="high"`。

In [ ]:
prompt = """
Create a simple biology diagram titled "Cellular Respiration at a Glance" for high school students.

Show how glucose turns into energy inside a cell. Include glycolysis, the Krebs cycle, and the electron transport chain.
Use arrows to connect the steps, and label the main molecules: glucose, pyruvate, ATP, NADH, FADH2, CO2, O2, and H2O.
Make it look like a clean classroom handout or slide, with a white background, simple icons, clear labels, and easy-to-read text.

Avoid tiny text, extra decoration, or anything that makes the diagram hard to understand.
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1536x1024",
    quality="high",
)

save_image(result, "scientific_educational_cellular_respiration_gpt-image-2.png")

输出图像：

![](../../images/scientific_educational_cellular_respiration_gpt-image-2.png)

## 4.10 幻灯片、示意图、图表和生产力类图像

当提示写得像一份成果物规格说明，而不是插画请求时，生产力类视觉内容的效果最好。指明确切的交付物（幻灯片、工作流示意图、图表、页面图像），定义画布和层级，提供真实的文字或数据，并描述视觉语言。这类提示应包含实际约束：字体易读、间距精致、没有装饰性杂物，也不要采用千篇一律的图库照片风格。

对于幻灯片、图表和以示意图为主的资产，请直接在提示中写明数字和标签。演示文稿类输出使用横版尺寸；当图像包含小号文字、图例、坐标轴或脚注时，使用 `quality="high"`。

In [ ]:
prompt = """
Create one pitch-deck slide titled **"Market Opportunity"** that feels like a real Series A fundraising slide from a YC-backed startup.

Use a clean white background, modern sans-serif typography like Inter, and a crisp, minimal layout. The slide should include:

* A TAM/SAM/SOM concentric-circle diagram in muted blues and grays
* Specific, believable market sizing numbers:

  * **TAM:** $42B
  * **SAM:** $8.7B
  * **SOM:** $340M
* A clean bar chart below showing market growth from **2021 to 2026**, with a subtle upward trend
* Small footnotes: **"AGI Research, 2024"** and **"Internal analysis"**
* A company logo placeholder in the bottom-right corner

The design should look like it belongs in a deck that actually raised money: highly readable text, clear data hierarchy, polished spacing, and professional startup-style visual language.

Avoid clip art, stock photography, gradients, shadows, decorative elements, or anything that feels generic or overdesigned.
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1536x864",
    quality="high",
)

save_image(result, "market_opportunity_slide_gpt-image-2.png")

输出图像：

![](../../images/market_opportunity_slide_gpt-image-2.png)

## 5. 用例——编辑（文本 + 图像 → 图像）

## 5.1 风格迁移

当你希望保留参考图像的*视觉语言*（调色板、纹理、笔触、胶片颗粒等），同时改变主体或场景时，风格迁移非常有用。为获得最佳效果，请描述哪些必须保持一致（风格线索）、哪些必须改变（新内容），并添加背景、取景和“no extra elements”等硬性约束，以防止漂移。

In [ ]:
prompt = """
Use the same style from the input image and generate a man riding a motorcycle on a white background.
"""

result = client.images.edit(
    model="gpt-image-2",
    image=[
        open("../../images/input_images/pixels.png", "rb"),
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "motorcycle_gpt-image-2.png")

输入图像：

![](../../images/pixels.png)

输出图像：

![](../../images/motorcycle_gpt-image-2.png)

## 5.2 虚拟试衣


虚拟试衣非常适合对身份保持要求极高的电商预览。关键是明确锁定人物（面部、体型、姿势、发型、表情），*只*允许改变服装，然后要求真实的穿着效果（垂坠、褶皱、遮挡）以及一致的光照/阴影，使服装看起来是自然穿在身上的，而不是贴上去的。

In [ ]:
prompt = """
Edit the image to dress the woman using the provided clothing images. Do not change her face, facial features, skin tone, body shape, pose, or identity in any way. Preserve her exact likeness, expression, hairstyle, and proportions. Replace only the clothing, fitting the garments naturally to her existing pose and body geometry with realistic fabric behavior. Match lighting, shadows, and color temperature to the original photo so the outfit integrates photorealistically, without looking pasted on. Do not change the background, camera angle, framing, or image quality, and do not add accessories, text, logos, or watermarks.
"""

result = client.images.edit(
    model="gpt-image-2",
    image=[
        open("../../images/input_images/woman_in_museum.png", "rb"),
        open("../../images/input_images/tank_top.png", "rb"),
        open("../../images/input_images/jacket.png", "rb"),
        open("../../images/input_images/tank_top.png", "rb"),
        open("../../images/input_images/boots.png", "rb"),
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "outfit_gpt-image-2.png")

输入图像：

| 全身照 | 单品 1 |
|:------------:|:--------------:|
| ![](../../images/woman_in_museum.png)  | ![](../../images/jacket.png) |
| 单品 2 | 单品 3 |
| ![](../../images/tank_top.png) | ![](../../images/boots.png) |

输出图像：

![](../../images/outfit_gpt-image-2.png)

## 5.3 草图 → 图像（渲染）


草图转渲染工作流非常适合将粗略的手绘转化为照片级写实的概念图，同时保留原始意图。把提示当作规格说明来写：保留布局和透视，然后通过指定合理的材质、光照和环境来*增加真实感*。加入“do not add new elements/text”，以避免模型进行创造性的重新诠释。

In [ ]:
prompt = """
Turn this drawing into a photorealistic image.
Preserve the exact layout, proportions, and perspective.
Choose realistic materials and lighting consistent with the sketch intent.
Do not add new elements or text.
"""

result = client.images.edit(
    model="gpt-image-2",
    image=[
        open("../../images/input_images/drawings.png", "rb"),
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "realistic_valley_gpt-image-2.png")

输入图像：

![](../../images/drawings.png)

输出图像：

![](../../images/realistic_valley_gpt-image-2.png)

## 5.4 产品样机（透明背景 + 标签完整性）


产品抠图和样机准备常用于商品目录、电商平台和设计系统。成功与否取决于边缘质量（轮廓干净，没有色边/光晕）和标签完整性（文字保持清晰且不被改动）。`gpt-image-2` 的透明背景功能处于预览阶段；请求 `background="transparent"` 并搭配 `output_format="png"`（默认值）或 `output_format="webp"`，即可直接生成可复用的产品抠图。`jpeg` 不支持透明背景。要求呈现独立主体，保留现有的产品几何形状和标签，并省略纯色背景、棋盘格和不必要的阴影。PNG 输出时省略 `output_compression`；WebP 支持可选压缩。

In [ ]:
prompt = """
Extract the product from the input image and isolate it on a fully transparent background.
Output: centered product, crisp silhouette, no halos/fringing.
Preserve product geometry and label legibility exactly.
Add only light polishing. Do not add a solid backdrop, checkerboard, scenery, or shadow.
Do not restyle the product; remove the background and preserve clean alpha transparency.
"""

result = client.images.edit(
    model="gpt-image-2",
    image=[
        open("../../images/input_images/shampoo.png", "rb"),
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
    background="transparent",
    output_format="png",
)

save_image(result, "extract_product_gpt-image-2.png")

输入图像：

![](../../images/shampoo.png)

输出图像：

![](../../images/extract_product_gpt-image-2.png)

## 5.5 带有真实图中文字的营销创意


带有真实图中文字的营销创意非常适合快速构思广告概念，但字体排印需要明确的约束。将准确的文案放在引号中，要求逐字呈现（不得有多余字符），并描述位置和字体风格。如果文字保真度不够理想，请保持提示严格并持续迭代——对措辞/布局做些小调整通常就能提高可读性。

In [ ]:
prompt = """
Create a realistic billboard mockup of the shampoo on a highway scene during sunset.
Billboard text (EXACT, verbatim, no extra characters):
"Fresh and clean"
Typography: bold sans-serif, high contrast, centered, clean kerning.
Ensure text appears once and is perfectly legible.
No watermarks, no logos.
"""

result = client.images.edit(
    model="gpt-image-2",
    image=[
        open("../../images/input_images/shampoo.png", "rb"),
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "billboard_gpt-image-2.png")

输入图像：

![](../../images/shampoo.png)

输出图像：

![](../../images/billboard_gpt-image-2.png)

## 5.6 光照与天气转换


用于为照片重新布景，以呈现不同的氛围、季节或一天中不同时间的变体（例如晴天 → 阴天、白天 → 黄昏、晴朗 → 下雪），同时保持场景构图不变。关键是只改变环境条件——光线方向/质感、阴影、大气、降水和地面湿度——同时保留身份、几何结构、相机角度和物体位置，使其看起来仍是同一张原始照片。

In [ ]:
prompt = """
Make it look like a winter evening with snowfall.
"""

result = client.images.edit(
    model="gpt-image-2",
    input_fidelity="high", 
    image=[
        open("../../images/output_images/billboard_gpt-image-2.png", "rb"),
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "billboard_winter_gpt-image-2.png")


输出图像： 

![](../../images/billboard_winter_gpt-image-2.png)


## 5.7 物体移除


人物入景合成适用于故事板、营销活动以及需要保持面部/身份的“假设”场景。通过指定贴近真实的摄影观感（自然光照、可信的细节、不做电影感调色）来锚定真实感，并锁定主体上不得改变的部分。在可用时，更高的输入保真度有助于在较大的场景编辑中保持相似度。

In [ ]:
prompt = """
Remove the flower from man's hand. Do not change anything else.
"""


result = client.images.edit(
    model="gpt-image-2",
    input_fidelity="high", 
    image=[
        open("../../images/output_images/man_with_blue_hat.png", "rb"),
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "man_with_no_flower_gpt-image-2.png")

输入与输出图像：

| 原始输入 | 输出图像 |
|:------------:|:--------------:|
| ![](../../images/man_with_blue_hat.png)  |  ![](../../images/man_with_no_flower_gpt-image-2.png) |


## 5.8 将人物插入场景


人物入景合成适用于故事板、营销活动以及需要保持面部/身份的“假设”场景。通过指定贴近真实的摄影观感（自然光照、可信的细节、不做电影感调色）来锚定真实感，并锁定主体上不得改变的部分。在可用时，更高的输入保真度有助于在较大的场景编辑中保持相似度。

In [ ]:
prompt = """
Generate a highly realistic action scene where this person is running away from a large, realistic brown bear attacking a campsite. The image should look like a real photograph someone could have taken, not an overly enhanced or cinematic movie-poster image.
She is centered in the image but looking away from the camera, wearing outdoorsy camping attire, with dirt on her face and tears in her clothing. She is clearly afraid but focused on escaping, running away from the bear as it destroys the campsite behind her.
The campsite is in Yosemite National Park, with believable natural details. The time of day is dusk, with natural lighting and realistic colors. Everything should feel grounded, authentic, and unstyled, as if captured in a real moment. Avoid cinematic lighting, dramatic color grading, or stylized composition.
"""

result = client.images.edit(
    model="gpt-image-2",
    input_fidelity="high", 
    image=[
        open("../../images/input_images/woman_in_museum.png", "rb"),
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "scene_gpt-image-2.png")

输出图像：


In [ ]:
from IPython.display import Image, display
display(Image(filename="../../images/output_images/scene_gpt-image-2.png", width=500))

## 5.9 多图像引用与合成


用于将多个输入中的元素组合成一张可信的图像——非常适合“将这个物体/人物插入那个场景”的工作流，而无需重新生成全部内容。关键是清楚说明要移植什么（图像 2 中的狗）、放到哪里（紧挨着图像 1 中的女人），以及哪些必须保持不变（场景、背景、取景），同时匹配光照、透视、比例和阴影，使合成结果看起来就像是在原始照片中自然拍摄到的。

In [ ]:
prompt = """
Place the dog from the second image into the setting of image 1, right next to the woman, use the same style of lighting, composition and background. Do not change anything else.
"""

result = client.images.edit(
    model="gpt-image-2",
    input_fidelity="high", 
    image=[
        open("../../images/output_images/test_woman.png", "rb"),
        open("../../images/output_images/test_woman_2.png", "rb"),
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)


save_image(result, "test_woman_with_dog_gpt-image-2.png")

输入与输出图像：

| 原始输入 | 移除红色条纹 | 更改帽子颜色 |
|:--------------:|:------------------:|:----------------:|
| ![](../../images/test_woman.png) | ![](../../images/test_woman_2.png) | ![](../../images/test_woman_with_dog_gpt-image-2.png) |

## 6. 其他高价值用例

## 6.1 室内设计“替换”（精确编辑）
用于在真实空间中可视化家具或装饰的变化，而无需重新渲染整个场景。目标是精确到位的真实感：只替换单个物体，同时保留相机角度、光照、阴影和周围环境，使编辑结果看起来像一张真实照片，而不是重新设计。

In [ ]:
prompt = """
In this room photo, replace ONLY white with chairs made of wood.
Preserve camera angle, room lighting, floor shadows, and surrounding objects.
Keep all other aspects of the image unchanged.
Photorealistic contact shadows and fabric texture.
"""

result = client.images.edit(
    model="gpt-image-2",
    image=[
        open("../../images/input_images/kitchen.jpeg", "rb"),
    ],
    prompt=prompt,
    size="1536x1024",
    quality="medium",
)

save_image(result, "kitchen-chairs_gpt-image-2.png")

输入与输出图像：

| 输入图像 | 输出图像 |
|------------|--------------|
| ![](../../images/kitchen.jpeg) | ![](../../images/kitchen-chairs_gpt-image-2.png) |

## 6.2 3D 立体节日贺卡（产品风格样机）
非常适合季节性营销概念和印刷预览。强调触感上的真实感——纸张层次、纤维、折痕和柔和的影棚光——使结果看起来像一件被拍摄的实体产品，而不是一幅平面插画。

In [ ]:
scene_description = (
    "a cozy Christmas scene with an old teddy bear sitting inside a keepsake box, "
    "slightly worn fur, soft stitching repairs, placed near a window with falling snow outside. "
    "The scene suggests the child has grown up, but the memories remain."
)

short_copy = "Merry Christmas — some memories never fade."

prompt = f"""
Create a Christmas holiday card illustration.

Scene:
{scene_description}

Mood:
Warm, nostalgic, gentle, emotional.

Style:
Premium holiday card photography, soft cinematic lighting,
realistic textures, shallow depth of field,
tasteful bokeh lights, high print-quality composition.

Constraints:
- Original artwork only
- No trademarks
- No watermarks
- No logos

Include ONLY this card text (verbatim):
"{short_copy}"
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "christmas_holiday_card_teddy_gpt-image-2.png")

输出图像：

![](../../images/christmas_holiday_card_teddy_gpt-image-2.png)

## 6.3 收藏级可动人偶 / 毛绒钥匙扣（周边概念）

用于早期周边构思和提案视觉。侧重于高端产品摄影的线索（材质、包装、印刷清晰度），同时保持设计原创且不侵权。非常适合快速测试多种角色或包装变体。


In [ ]:
# ---- Inputs ----
character_description = (
    "a vintage-style toy propeller airplane with rounded wings, "
    "a front-mounted spinning propeller, slightly worn paint edges, "
    "classic childhood proportions, designed as a nostalgic holiday collectible"
)

short_copy = "Christmas Memories Edition"

# ---- Prompt ----
prompt = f"""
Create a collectible action figure of {character_description}, in blister packaging.

Concept:
A nostalgic holiday collectible inspired by the simple toy airplanes
children used to play with during winter holidays.
Evokes warmth, imagination, and childhood wonder.

Style:
Premium toy photography, realistic plastic and painted metal textures,
studio lighting, shallow depth of field,
sharp label printing, high-end retail presentation.

Constraints:
- Original design only
- No trademarks
- No watermarks
- No logos

Include ONLY this packaging text (verbatim):
"{short_copy}"
"""

result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "christmas_collectible_toy_airplane_gpt-image-2.png")

输出图像： 

![](../../images/christmas_collectible_toy_airplane_gpt-image-2.png)

## 6.4 保持角色一致性的儿童绘本插画（多图像工作流）
专为不能容忍角色漂移的多页插画流水线而设计。一个可复用的“角色锚点”能确保在不同场景、姿势和页面之间保持视觉连贯，同时允许环境和情节的变化。


1️⃣ 角色锚点——确立可复用的主角

目标：锁定角色的外貌、比例、服装和基调。

In [ ]:
# ---- Inputs ----
prompt = """
Create a children’s book illustration introducing a main character.

Character:
A young, storybook-style hero inspired by a little forest outlaw,
wearing a simple green hooded tunic, soft brown boots, and a small belt pouch.
The character has a kind expression, gentle eyes, and a brave but warm demeanor.
Carries a small wooden bow used only for helping, never harming.

Theme:
The character protects and rescues small forest animals like squirrels, birds, and rabbits.

Style:
Children’s book illustration, hand-painted watercolor look,
soft outlines, warm earthy colors, whimsical and friendly.
Proportions suitable for picture books (slightly oversized head, expressive face).

Constraints:
- Original character (no copyrighted characters)
- No text
- No watermarks
- Plain forest background to clearly showcase the character
"""

# ---- Image generation ----
result = client.images.generate(
    model="gpt-image-2",
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "childrens_book_illustration_1_gpt-image-2.png")

输出图像：

![](../../images/childrens_book_illustration_1_gpt-image-2.png)

2️⃣ 故事延续——复用角色，推进情节

目标：同一个角色，新的场景 + 动作。
角色外貌必须保持不变。

In [ ]:
# ---- Inputs ----
prompt = """
Continue the children’s book story using the same character.

Scene:
The same young forest hero is gently helping a frightened squirrel
out of a fallen tree after a winter storm.
The character kneels beside the squirrel, offering reassurance.

Character Consistency:
- Same green hooded tunic
- Same facial features, proportions, and color palette
- Same gentle, heroic personality

Style:
Children’s book watercolor illustration,
soft lighting, snowy forest environment,
warm and comforting mood.

Constraints:
- Do not redesign the character
- No text
- No watermarks
"""
# ---- Image generation ----
result = client.images.edit(
    model="gpt-image-2",
    image=[
        open("../../images/output_images/childrens_book_illustration_1_gpt-image-2.png", "rb"),  # use image from step 1
    ],
    prompt=prompt,
    size="1024x1536",
    quality="medium",
)

save_image(result, "childrens_book_illustration_2_gpt-image-2.png")


输出图像：

![](../../images/childrens_book_illustration_2_gpt-image-2.png)

## 结论 

在本 notebook 中，我们演示了如何使用 gpt-image 生成模型构建高质量、可控、能在真实生产环境中经受考验的图像生成和编辑工作流。本 cookbook 强调将提示结构、明确的约束和小步迭代作为控制真实感、布局、文字准确性和身份保持的主要手段。我们涵盖了生成和编辑两类模式，从信息图、照片级写实、UI 原型和 logo，到翻译、风格迁移、虚拟试衣、合成和光照变化。在所有示例中，本 cookbook 反复强调要清晰区分应当改变的内容与必须保持不变的内容，并在每次迭代中重申这些不变项以防止漂移。我们还强调了质量和输入保真度设置如何让你根据用例在延迟与视觉精度之间做出有意识的权衡。这些示例共同构成了一套在生产图像工作流中应用 gpt-image 生成模型的实用、可复用的操作手册。